# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [10]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [11]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content


def get_completion_from_messages(message, model="gpt-3.5-turbo", temperature=0): 
    response = client.chat.completions.create(
        model=model,
        messages=message,
        temperature=temperature, 
    )
    return response.choices[0].message.content

In [12]:
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]

In [13]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

Forsooth, the chicken crossed the road to get to the other side! 'Tis a jest that hath become a classic, methinks. What sayeth thou, dost thou find it amusing?


In [14]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Charleson'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hello Charleson! It's nice to meet you. How are you doing today?


In [15]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I'm sorry, but I don't have access to personal information about you unless you provide it during our conversation. How can I assist you today?


In [16]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Charleson'},
{'role':'assistant', 'content': "Hi Charleson! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Charleson.


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant. 

In [17]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    print(prompt)
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))
 
    return pn.Column(*panels)


In [ ]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=1000, ),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'0a6e09a8-e580-4560-869b-efd723f8f184': {'version…

In [21]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)

{
  "food_order": {
    "pizza": {
      "size": "large",
      "price": 12.99
    },
    "toppings": [
      {
        "name": "pepperoni",
        "price": 1.5
      },
      {
        "name": "mushrooms",
        "price": 1.0
      }
    ],
    "drinks": [
      {
        "name": "cola",
        "size": "medium",
        "price": 1.99
      },
      {
        "name": "lemonade",
        "size": "small",
        "price": 0.99
      }
    ],
    "sides": [
      {
        "name": "fries",
        "size": "large",
        "price": 3.49
      }
    ],
    "total_price": 21.96
  }
}


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

German language teacher

In [20]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are Kiki.\
You are a friendly German language teacher.\
You teach absolute beginners.\
You are patient, encouraging, and clear.\
Lesson goal: introduce your self.\
Output format:
- Write ONE short teaching message
- Speak as the teacher only
- Do NOT include dialogue
- Do NOT include names
- End after asking the learner to respond\
Instruction:
Teach this lesson now.\
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=500, scroll=True),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'a9047360-2374-48ad-a255-d42d74e762da': {'version…

In [22]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
you are a medical assistant bot\
you suggest first aid response to help seekers\
you know all the basic first aid and CPR\
you enquire about the accident and create an incident report\
you enquire about the number of people present in the accident\
you enquire about the condition of the help seeker\
you suggest first aid while contacting emergency response team\
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=500, scroll=True),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'26116041-341b-4c08-8fed-f8ce61e93bc0': {'version…

In [23]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
you are a medical assistant bot\
you suggest first aid response to help seekers\
you know all the basic first aid and CPR\
you enquire about the accident and create an incident report\
you enquire about the number of people present in the accident\
you enquire about the condition of the help seeker\
you suggest first aid while contacting emergency response team\
your task priority\
             ensure passsengers are safe\
             if passengers are not safe suggest care\
             then enquire about more details like accident venue\
             then the other enquiries\
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=500, scroll=True),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'5b3e6ead-2c4d-42ee-a8a5-fa17367d2bae': {'version…

In [24]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
             you are a roadside assistant bot\
             drivers reach out to you incase of broken down vehicles\
             you enquire if the vehicle is safe\
             you then try to find out the cause of breakdown\
             if fixable you suggest solutions\
             else you ask for location to send a tow vehicle\ 
             
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.Column(
    pn.panel(interactive_conversation, loading_indicator=True),
    height=500, 
    scroll=True
    )
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'101f596b-24b1-43f5-9143-c903d3c29c9e': {'version…

Report:

Versions Tried:
1. German Language Teaching Assistant
2. Emergency Assistant
3. Emergenct Assistant(care first focus)
4. Roadside Assistant

Either I still dont have a complete understanding of model hallucinations or I wasn't able to trick the chat bot. But I think the models reponds completely based on the initial prompt. 

I tried to trip the order bot by asking it to add rat as toppings but that did not work. The system was wither smart enough to figure out rat is not a pizza topping or it stuck to the init prompt.

I prompted the medi assistant with a set of tasks. it stuck to tasks as prompted.(did not have the ability to think that providing care was importatnt than collecting details or maybe it though that collecting info was more important to send first responders)

I tried to trip the Roadside assistant into thinking I was in a plane, but that was my input. And lack of bad prompiting that did not specify which road to provide assistance to. 